# Introduction to Hugging Face
Hugging Face provides state-of-the-art NLP (and increasingly,

multimodal) models and an easy-to-use Python library called Transformers.

With just a few lines of code you can:

Load pre-trained models for dozens of tasks
Run inference via high-level “pipelines”
Fine-tune on your own data

# 1. Setup in Google Colab

In [1]:
# Install the core libraries
!pip install transformers datasets huggingface_hub --quiet

# 2. Quick-start with Pipelines

## Sentiment_Anlysis

In [2]:
from transformers import pipeline

# 1. Load a sentiment-analysis pipeline (defaults to a small model)
sentiment = pipeline("sentiment-analysis")

# 2. Run inference
examples = [
    "Hugging Face makes NLP super accessible!",
    "I dislike bugs in my code..."
]
results = sentiment(examples)
for text, res in zip(examples, results):
    print(f"{text!r:50} → label={res['label']}, score={res['score']:.3f}")

[transformers] No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

'Hugging Face makes NLP super accessible!'         → label=POSITIVE, score=1.000
'I dislike bugs in my code...'                     → label=NEGATIVE, score=0.999


## Text_Generation

In [3]:
from transformers import pipeline

generator = pipeline("text-generation", model="gpt2")
prompt = "Once upon a time"
out = generator(prompt, max_length=30, num_return_sequences=1)
print(out[0]["generated_text"])

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'num_return_sequences', 'max_length'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=256) and `max_length`(=30) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer GPT2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


Once upon a time, the entire country would have to pass through the gates of the capital, and the people would be so much less likely to take to the streets.

The government had no interest in giving up the people's freedom, and the people were just a bunch of people, they were all free.

The people of the country, when they realized that this was going on, were so very, very, very angry and were so very angry at the government, they began to try and form an alliance with the other countries, and, of course, they tried to make a deal with the other countries, and it worked. The government finally gave up the control of the capital, and the people were so much more likely to go to the streets.

The people of the country were so much more likely to fight back against the government.

The people of the country had no desire to have an ally. They were all afraid of the government. They thought that they would be the enemies of the entire country.

The people of the country were so much mor

# Finetuning GPT-2

# Problem 1 – GPT-2 Fine-Tuning for Sentiment Analysis

In [4]:
!pip install -U datasets huggingface_hub

## Dataset + Tokenizer

In [5]:
from datasets import load_dataset
from transformers import GPT2Tokenizer

# Load IMDb directly from Hugging Face
dataset = load_dataset("stanfordnlp/imdb")

# GPT-2 tokenizer
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")

# GPT-2 does not have a pad token by default
tokenizer.pad_token = tokenizer.eos_token

print(dataset)

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})


## Tokkenise dataset

In [6]:
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

tokenized_dataset = dataset.map(
    tokenize_function,
    batched=True
)

print(tokenized_dataset)

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/50000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label', 'input_ids', 'attention_mask'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label', 'input_ids', 'attention_mask'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label', 'input_ids', 'attention_mask'],
        num_rows: 50000
    })
})


## GPT-2

In [7]:
from transformers import GPT2ForSequenceClassification

model = GPT2ForSequenceClassification.from_pretrained(
    "gpt2",
    num_labels=2
)

model.config.pad_token_id = tokenizer.pad_token_id

print("Model loaded successfully!")

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

[transformers] GPT2ForSequenceClassification LOAD REPORT from: gpt2
Key          | Status  | 
-------------+---------+-
score.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model loaded successfully!


# Training

In [10]:
!pip install -U transformers accelerate

In [11]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./gpt2-imdb",
    num_train_epochs=2,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=100,
    learning_rate=5e-5,
    weight_decay=0.01,
    report_to="none"
)

# Trainer

In [13]:
!pip install -U transformers accelerate

In [15]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"]
)

In [16]:
trainer.train()

Epoch,Training Loss,Validation Loss
1,0.331440,0.311021
2,0.223398,0.365824


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=6250, training_loss=0.31418488037109377, metrics={'train_runtime': 1779.5246, 'train_samples_per_second': 28.097, 'train_steps_per_second': 3.512, 'total_flos': 3266209382400000.0, 'train_loss': 0.31418488037109377, 'epoch': 2.0})

# Evaluation

In [17]:
results = trainer.evaluate()
print(results)


Training Loss,Validation Loss,Epoch
0.223398,0.365824,2


{'eval_loss': 0.36582428216934204}


# Model Testing

In [18]:
import torch

def predict_sentiment(text):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=128
    )

    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)

    prediction = torch.argmax(outputs.logits, dim=-1).item()

    return "Positive" if prediction == 1 else "Negative"

In [19]:
reviews = [
    "This movie was absolutely amazing. I loved every moment of it!",
    "The movie was boring, poorly written, and a complete waste of time."
]

for review in reviews:
    print("Review:", review)
    print("Prediction:", predict_sentiment(review))
    print()

Review: This movie was absolutely amazing. I loved every moment of it!
Prediction: Positive

Review: The movie was boring, poorly written, and a complete waste of time.
Prediction: Negative

